In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

### 🎯MP3 음성을 인식해 텍스트로 변환시키기

In [2]:
audio_file_path = 'audio/lsy_audio_2023_58s.mp3' # MP3 파일 경로 입력

with open(audio_file_path, 'rb') as audio_file:
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
    )

In [3]:
from rich.pretty import pprint
pprint(transcription)

Transcription(
│   text='안녕하세요. 이 강의는 GPT API로 챗봇 만들기라는 내용을 다루는 강의입니다. GPT API에 대해서 생소하신 분들도 있을 텐데 우리가 잘 알고 있는 채GPT, 채GPT 기능을 이용해서 우리가 원하는 프로그램을 어떻게 만드는지에 대해서 이야기할 거예요. 그래서 이런 강의들이 사실 많이 있습니다. 그래서 여러 가지들이 있는데 좀 이 강의의 특징이라고 한다면 GPT로 명확한 미션을 달성하는 챗봇 프로그램을 만드는 게 사실 쉽지는 않은데 이걸 어떻게 해서 구현을 하는지 그리고 그게 왜 필요한지에 대해서 좀 이야기를 할 거고요. 그 예제로 예제는 여러 가지가 될 수 있는데 여기서 예제로 하는 것은 음악 플레이리스트 동영상을 자동으로 대화를 통해서 생성하는 프로그램 만드는 것을 다루려고 합니다. 그래서 프로그램이 실행되는 모습을 한번 보여드릴게요. 우리가 만들 프로그램은 이런 식으로 이제 나타나게 되고요.',
│   languages=None,
│   logprobs=None,
│   usage=UsageDuration(seconds=58.0, type='duration')
)

#### 💡한국어 음성 파일을 영어로 번역하기

In [4]:
with open(audio_file_path, 'rb') as audio_file:
    translation = client.audio.translations.create(
        model="whisper-1",
        file=audio_file
    )
pprint(translation)

Translation(
│   text="Hello, this is a lecture on how to make a chatbot with GPT API. Some of you may be unfamiliar with GPT API. We're going to talk about how to make the program we want using the chat GPT function that we know well. So there are a lot of lectures like this. There are many things, but if I were to say the characteristics of this lecture, it's not easy to make a chatbot program that achieves a clear mission with GPT. I'm going to talk about how to implement this and why it's necessary. As an example, there can be many examples. The example here is to create a program that automatically creates a music playlist video through conversation. So let me show you how the program runs. The program we're going to make is going to look like this."
)

##### ✏️ gpt-4o-transcribe-diarize 모델 사용해보기
주요 특징 및 기능
- 화자 분리(Speaker Diarization): 대화에서 누가 언제 말했는지 식별하여 각 발화(segment)에 스피커 라벨과 시작·끝 시간 정보를 제공합니다.
- 높은 정확도: 기존 Whisper 모델보다 단어 오류율(WER)이 낮고 언어 인식 성능이 뛰어납니다.
- API 전용 지원: 음성 인식(Transcription) API에서만 사용할 수 있는 모델입니다

In [10]:
import json

audio_file_path = 'audio/싼기타_비싼기타.mp3'

with open(audio_file_path, 'rb') as audio_file:
    response = client.audio.transcriptions.create(
        model="gpt-4o-transcribe-diarize",
        file=audio_file,
        response_format="diarized_json",  # 화자 정보를 받기 위해 필수 지정
        chunking_strategy="auto"         # 30초 이상 긴 파일의 화자 분리 정확도 향상
    )

# 응답 결과 출력 (딕셔너리 형태로 변환하여 가독성 있게 출력)
response_dict = json.loads(response.model_dump_json())
pprint(response_dict)

{
│   'duration': 435.4873333333333,
│   'segments': [
│   │   {
│   │   │   'id': 'seg_0',
│   │   │   'end': 25.674000000000003,
│   │   │   'speaker': 'A',
│   │   │   'start': 0.724,
│   │   │   'text': ' 지금부터 저랑 역할극을 합시다 역할극을 스탠딩 코미디 스타일로 할 건데 토론을 하면서 자연스럽게 대화하는 형식으로 하면서 코미디를 진행해 봅시다 그래서 좀 재밌고 자연스럽고 유머러스하게 저랑 대화를 하시면 돼요 자연스럽게 그리고 주제는 그 싼 기타로 전기기타를 시작하는 게 좋으냐 아니면은 비싼 기타로 전기기타를 시작하는 게 좋으냐 요거를 입장을 나눠가지고',
│   │   │   'type': 'transcript.text.segment'
│   │   },
│   │   {
│   │   │   'id': 'seg_1',
│   │   │   'end': 30.122,
│   │   │   'speaker': 'A',
│   │   │   'start': 25.972,
│   │   │   'text': ' 저랑 토론해보면 좋을 것 같아요 둘 중에 어떤 역할 맡으실래요',
│   │   │   'type': 'transcript.text.segment'
│   │   },
│   │   {
│   │   │   'id': 'seg_2',
│   │   │   'end': 32.872,
│   │   │   'speaker': 'B',
│   │   │   'start': 32.222,
│   │   │   'text': ' 좋습니다.',
│   │   │   'type': 'transcript.text.segment'
│   │   },
│   │   {
│   │   │   'id': 'seg_3',
│   │   │   'end': 36.822,
│   │   │   'speaker': 'B',
│   │   │   'start': 33.272,
│   │   │   'text': ' 그럼 제가 쌍기타로 시작하는 게 좋다는 입장을 맡아볼게요.',
│   │   │   'type': 'transcript.text.segment'
│   │   },
│   │   {
│   │   │   'id': 'seg_4',
│   │   │   'end': 41.122,
│   │   │   'speaker': 'B',
│   │   │   'start': 37.522000000000006,
│   │   │   'text': ' 그럼 성혁님은 빗산기타로 시작하는 게 좋다는 입장이시죠?',
│   │   │   'type': 'transcript.text.segment'
│   │   },
│   │   {
│   │   │   'id': 'seg_5',
│   │   │   'end': 41.872,
│   │   │   'speaker': 'A',
│   │   │   'start': 41.47200000000001,
│   │   │   'text': ' 네 맞아요',
│   │   │   'type': 'transcript.text.segment'
│   │   },
│   │   {
│   │   │   'id': 'seg_6',
│   │   │   'end': 42.072,
│   │   │   'speaker': 'B',
│   │   │   'start': 41.872,
│   │   │   'text': ' 준비되셨나요?',
│   │   │   'type': 'transcript.text.segment'
│   │   },
│   │   {
│   │   │   'id': 'seg_7',
│   │   │   'end': 43.772000000000006,
│   │   │   'speaker': 'A',
│   │   │   'start': 42.072,
│   │   │   'text': ' 네 됐어요 시작하시죠',
│   │   │   'type': 'transcript.text.segment'
│   │   },
│   │   {
│   │   │   'id': 'seg_8',
│   │   │   'end': 46.022000000000006,
│   │   │   'speaker': 'B',
│   │   │   'start': 45.572,
│   │   │   'text': ' 좋아요.',
│   │   │   'type': 'transcript.text.segment'
│   │   },
│   │   {
│   │   │   'id': 'seg_9',
│   │   │   'end': 50.072,
│   │   │   'speaker': 'B',
│   │   │   'start': 46.672,
│   │   │   'text': ' 먼저 쌍기타로 시작하는 게 좋은 이유를 말씀드리겠습니다.',
│   │   │   'type': 'transcript.text.segment'
│   │   },
│   │   {
│   │   │   'id': 'seg_10',
│   │   │   'end': 59.254000000000005,
│   │   │   'speaker': 'B',
│   │   │   'start': 50.804,
│   │   │   'text': ' 초보자일 때는 실수도 많고 기타에 익숙해지는 과정이 필요하니까 비싼 기타보다는 부담 없이 연습할 수 있는 싼 기타가 좋다고 봅니다.',
│   │   │   'type': 'transcript.text.segment'
│   │   },
│   │   {
│   │   │   'id': 'seg_11',
│   │   │   'end': 64.654,
│   │   │   'speaker': 'B',
│   │   │   'start': 59.954,
│   │   │   'text': ' 무엇보다 실력이 향상되면 그때 좋은 기타로 업그레이드하는 것도 나쁘지 않죠?',
│   │   │   'type': 'transcript.text.segment'
│   │   },
│   │   {
│   │   │   'id': 'seg_12',
│   │   │   'end': 66.85,
│   │   │   'speaker': 'B',
│   │   │   'start': 65.3,
│   │   │   'text': ' 성형님은 어떻게 생각하세요?',
│   │   │   'type': 'transcript.text.segment'
│   │   },
│   │   {
│   │   │   'id': 'seg_13',
│   │   │   'end': 73.55,
│   │   │   'speaker': 'A',
│   │   │   'start': 66.89999999999999,
│   │   │   'text': ' 저는 지금 말에 어폐가 있다고 생각해요. 왜냐하면 어차피 지금 비싼 기타로 나중에 살 생각을 하는 거잖아요.',
│   │   │   'type': 'transcript.text.segment'
│   │   },
│   │   {
│   │   │   'id': 'seg_14',
│   │   │   'end': 82.5,
│   │   │   'speaker': 'A',
│   │   │   'start': 73.75,
│   │   │   'text': ' 그럼 어차피 비싼 기타도 사고 싼 기타도 사게 되는데 그런 이중 지출을 하지 말고 처음부터 비싼 기타로 가면 이중 지출을 막을 수가 있습니다.',
│   │   │   'type': 'transcript.text.segment'
│   │   },
│   │   {
│   │   │   'id': 'seg_15',
│   │   │   'end': 90.75,
│   │   │   'speaker': 'B',
│   │   │   'start': 84.25,
│   │   │   'text': ' 그런데 비싼 기타로 시작하면 혹시라도 흠집이 나거나 실수할 때 부담이 더 크지 않을까요?',